# Week 6 Period-Specific Isolation Forest — Huaye

This notebook is the authoritative Huaye Week 6 Isolation Forest analysis. It fits six independent models—one for each operating period—and uses only the six physical 15-minute rolling features. The earlier global 42-feature model and fixed top-percentage labels have been retired from the current implementation.

## Method

The dataset is split into `night_off`, `ramp_up`, `core_work`, `ramp_down`, `evening_off`, and `weekend`. A separate `IsolationForest` is fitted within each period using `contamination='auto'`. A row is labelled as a candidate anomaly only when that period model returns `predict(X) == -1`, equivalently `decision_function(X) < 0`. No fixed 3%, 5%, or 9% anomaly quota is imposed.

The output is an unsupervised candidate list, not confirmed faults. Baseline and LOF labels are retained only for descriptive comparison and are never model inputs.

In [1]:
from pathlib import Path
import sys
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'isolation_forest_period_auto_huaye.py').is_file():
    candidate = NOTEBOOK_DIR / 'ml'
    if (candidate / 'isolation_forest_period_auto_huaye.py').is_file():
        NOTEBOOK_DIR = candidate
    else:
        raise FileNotFoundError('Cannot locate isolation_forest_period_auto_huaye.py')
sys.path.insert(0, str(NOTEBOOK_DIR))

from isolation_forest_period_auto_huaye import (
    MODEL_FEATURES,
    MODEL_PARAMETERS,
    OUTPUT_DIR,
    OUTPUT_FILES,
    PERIOD_ORDER,
    run_pipeline,
)

print('Operating periods:', PERIOD_ORDER)
print('Physical features:', len(MODEL_FEATURES))
print('Model parameters:', MODEL_PARAMETERS)

Operating periods: ['night_off', 'ramp_up', 'core_work', 'ramp_down', 'evening_off', 'weekend']
Physical features: 6
Model parameters: {'n_estimators': 100, 'max_samples': 'auto', 'contamination': 'auto', 'max_features': 1.0, 'bootstrap': False, 'random_state': 42, 'n_jobs': -1}


## Fit Six Period Models and Save Outputs

The implementation preserves `timestamp + floor + zone`, aligns the unscaled physical rolling values and Gavin's LOF result by key, fits each period once, validates the automatic decision threshold, and writes full and anomaly-only outputs.

In [2]:
results, summaries, modelling_summary = run_pipeline()
print(f"Rows: {len(results):,}")
print(f"Candidate anomalies: {modelling_summary['anomaly_count']:,}")
print(f"Candidate anomaly rate: {modelling_summary['anomaly_rate_pct']:.4f}%")

Rows: 362,864
Candidate anomalies: 54,419
Candidate anomaly rate: 14.9971%


## Results by Operating Period

The anomaly proportion is allowed to vary by period. These percentages are produced by each model's automatic decision boundary; they are not preselected quotas and are not estimates of confirmed fault prevalence.

In [3]:
period_summary = summaries['period_summary'].copy()
period_summary[['operating_period', 'rows', 'anomaly_count', 'anomaly_rate_pct', 'lof_overlap', 'baseline_overlap']]

,operating_period,rows,anomaly_count,anomaly_rate_pct,lof_overlap,baseline_overlap
0,night_off,75600,5980,7.910053,579,824
1,ramp_up,10800,2455,22.731481,340,298
2,core_work,97200,17430,17.932099,1401,7884
3,ramp_down,21600,3010,13.935185,169,1564
4,evening_off,54000,11559,21.405556,1029,1385
5,weekend,103664,13985,13.490701,1115,2012


## Concrete Candidate Anomaly Values

The anomaly-only output contains the physical readings, unscaled 15-minute rolling values, scaled model inputs, anomaly score, decision function, and comparison labels. Higher `period_if_anomaly_score` means more unusual within the same operating period. Raw scores from separate period models should not be compared directly across periods.

In [4]:
example_columns = [
    'timestamp', 'floor', 'zone', 'operating_period',
    'total_AC_power_kw', 'light_power_kw', 'plug_power_kw',
    'temperature_c', 'humidity_pct', 'lux',
    'period_if_anomaly_score', 'period_if_decision_function',
    'period_if_rank_within_period', 'lof_anomaly_label', 'baseline_anomaly_label',
]
(
    results.loc[results['period_if_anomaly_label'].eq(1), example_columns]
    .sort_values(['operating_period', 'period_if_anomaly_score'], ascending=[True, False])
    .groupby('operating_period', observed=True)
    .head(3)
)

,timestamp,floor,zone,operating_period,total_AC_power_kw,light_power_kw,plug_power_kw,temperature_c,humidity_pct,lux,period_if_anomaly_score,period_if_decision_function,period_if_rank_within_period,lof_anomaly_label,baseline_anomaly_label
219352,2018-07-27 08:04:00,6,4,core_work,31.92,0.00,0.17,22.75,65.54,1.0,0.669088,-0.169088,1,1,1
219355,2018-07-27 08:07:00,6,4,core_work,31.84,0.00,0.04,22.66,65.23,1.0,0.660694,-0.160694,2,1,1
219353,2018-07-27 08:05:00,6,4,core_work,31.89,0.00,0.17,22.72,65.42,1.0,0.660237,-0.160237,3,1,1
133658,2018-07-30 19:46:00,6,2,evening_off,0.00,5.04,0.10,30.94,69.70,52.0,0.699522,-0.199522,1,0,1
133659,2018-07-30 19:47:00,6,2,evening_off,0.00,6.02,0.12,30.94,69.80,52.0,0.699522,-0.199522,2,0,1
133660,2018-07-30 19:48:00,6,2,evening_off,0.00,6.02,0.11,30.94,69.70,52.0,0.699522,-0.199522,3,0,1
236563,2018-08-08 06:55:00,6,4,night_off,1.44,3.40,0.11,26.53,67.40,23.0,0.783294,-0.283294,1,0,0
236562,2018-08-08 06:54:00,6,4,night_off,1.42,10.44,0.26,26.55,67.13,23.0,0.781705,-0.281705,2,0,0
236564,2018-08-08 06:56:00,6,4,night_off,1.44,3.40,0.17,26.51,66.94,23.0,0.781705,-0.281705,3,0,0
248690,2018-08-16 17:02:00,6,4,ramp_down,2.37,3.91,0.96,26.21,57.74,40.0,0.706011,-0.206011,1,0,0


## Descriptive Method Comparison

LOF still uses Gavin's existing fixed 0.05 contamination result, while the statistical baseline is a rule-based reference. Neither is ground truth, so overlap and Jaccard values do not measure accuracy.

In [5]:
summaries['comparison']

,comparison,if_anomaly_count,comparison_anomaly_count,overlap_count,union_count,jaccard_overlap,note
0,lof_k100_c050,54419,18143,4633,67929,0.068204,LOF uses its existing fixed contamination 0.05...
1,statistical_baseline_v3,54419,32777,13967,73229,0.190730,"Statistical baseline is a reference, not groun..."


## Validation

The checks below confirm row/key preservation, six independent period models, exactly six physical inputs, complete scores, consistent `predict` and `decision_function` labels, non-fixed period rates, and exclusion of comparison labels from training.

In [6]:
validation = pd.DataFrame(modelling_summary['validation'])
assert validation['passed'].all()
validation

,check,passed,evidence
0,row_count_preserved,True,"rows=362,864"
1,expected_row_count,True,"rows=362,864"
2,unique_row_key,True,timestamp + floor + zone
3,six_period_models,True,"models=['core_work', 'evening_off', 'night_off..."
4,six_physical_features,True,features=['numerical__total_AC_power_kw_rollin...
5,scores_complete,True,missing=0
6,decision_threshold_consistent,True,label=1 exactly when decision_function < 0
7,not_fixed_five_percent,True,"{'night_off': 7.910053, 'ramp_up': 22.731481, ..."
8,reference_labels_excluded,True,baseline and LOF labels are comparison-only
9,all_anomaly_rows_exportable,True,"anomalies=54,419"


## Generated Outputs

Only the new `if_period_auto_*` files are authoritative for Huaye's current Isolation Forest analysis. Presentation figures are intentionally outside the scope of this revision.

In [7]:
for name, filename in OUTPUT_FILES.items():
    print(f"{name}: {OUTPUT_DIR / filename}")

full_results: D:\实习任务\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye\if_period_auto_results_full.csv.gz
anomalies: D:\实习任务\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye\if_period_auto_anomalies.csv.gz
period_summary: D:\实习任务\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye\if_period_auto_summary.csv
zone_summary: D:\实习任务\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye\if_period_auto_summary_by_zone.csv
hour_summary: D:\实习任务\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye\if_period_auto_summary_by_hour.csv
comparison: 